Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* ResNet Included
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 100 blocks (MaxPool2D in each 20 block)

In [ ]:
out_channels = 8
size = 64

model_blocks = [
    nn.Conv2d(3, out_channels, 3, 1, 1),
    nn.BatchNorm2d(out_channels),
    nn.ReLU()
]

for stage in range(5):

    for i in range(20):

        conv = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        bn = nn.BatchNorm2d(out_channels)

        model_blocks.append(ResidualBlock([conv, bn, nn.ReLU()]))

    if stage < 3:
        model_blocks.append(nn.MaxPool2d(2, 2))
        size //= 2
    if stage<4:
        model_blocks.extend([
            nn.Conv2d(out_channels, out_channels*2, 3, 1, 1),
            nn.BatchNorm2d(out_channels*2),
            nn.ReLU()
    ])

        out_channels *= 2

print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(out_channels * size * size, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    )

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment7/",
    save_checkpoints=10,
    print_every=5
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment7/tables/training_metrics.csv", index=False)

### Training/Validation Trend (100 epochs)
* The model started with moderate learning capability, achieving 39.98% training accuracy and 50.66% validation accuracy at epoch 1, indicating that initial feature extraction was already effective.
* Training loss consistently decreased from 1.19 at epoch 1 to 0.07 by epoch 100, showing continuous improvement in fitting the training data.
* Training accuracy increased steadily throughout training, reaching above 90% after epoch 65 and 97.35% by the final epoch.
* Validation performance improved rapidly during early training, increasing from 50.66% accuracy at epoch 1 to around 70% accuracy by epoch 34.
* The model showed strong improvement between epochs 12 and 49, where validation F1 increased from 0.58 to above 0.74, indicating better class separation.
* The best stable validation performance occurred in the later training stage, especially around epochs 86, 97, 99, and 100, where validation F1 remained around 0.78–0.79.
* The model experienced several unstable validation periods, with sudden increases in validation loss despite improving training metrics.
* Large validation loss spikes occurred at epochs 54, 62, 68, 69, 83, 94, and 98, suggesting occasional prediction collapse or unstable optimization behavior.
* Despite high training performance, validation performance remained significantly lower, showing a generalization gap between training and unseen data.
* Training accuracy continued improving after validation accuracy had mostly stabilized, indicating increasing overfitting in the later epochs.
* The model reached its strongest generalization region after epoch 80, where validation accuracy remained consistently above 70% except for instability events.
* Validation precision remained generally higher than validation recall, showing that the model was more conservative in predictions and avoided some false positives.
* The confusion matrices show that the model learned all three classes reasonably well, but class confusion remained present, especially between the second and third classes.
* The model occasionally collapsed into predicting a dominant class, visible from confusion matrices where most samples were assigned to one class.
* Epoch 74 achieved the highest balanced validation performance before later instability, with validation accuracy of 78.10% and validation F1 of 78.17%.
* Epoch 99 achieved the highest validation F1 score overall among the stable final epochs, reaching 79.25%.
* Epoch 100 maintained strong validation performance while achieving the highest training performance, indicating the model was still learning but with some overfitting.
* The final model shows strong learning capacity but would benefit from regularization, early stopping, or learning rate scheduling to reduce validation instability.

The model demonstrated effective learning throughout the 100 epochs, with training loss continuously decreasing and training accuracy improving from approximately 40% to 97%. Validation performance improved significantly during training, reaching a stable high-performance region after approximately epoch 80. However, the increasing gap between training and validation metrics indicates overfitting in later epochs. The model experienced several sharp validation loss spikes caused by unstable prediction behaviour, although it recovered and achieved strong validation results. The best generalization performance was achieved around the late training stage, with validation accuracy close to 79% and validation F1 around 79%, while the final epochs mainly improved training performance rather than validation performance.

<b>Best Epoch 99</b>

<b>Loss</b>
* Train Loss = 0.10049680492287516
* Valid Loss = 0.8463770637413134

<b>Training Metrics</b>
* Train Accuracy = 0.9598298668861389
* Train Precison = 0.960106372833252
* Train Recall = 0.959850549697876
* Train F1 = 0.9599669575691223

<b>Validation Accuracy</b>
* Validation Accuracy = 0.7920354008674622
* Validation Precision = 0.7921906113624573
* Validation Recall = 0.7934367656707764
* Validation F1 = 0.7924633622169495

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

tester = Tester(
    model,
    test_loader,
    3,
    torch.nn.CrossEntropyLoss(),
    "cuda"
)

test_scores = tester.test_all_checkpoints(
    "../models/experiment7"
)

### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment7/tables/test_metrics.csv", index=False)

### Test Performance Trend
* The model improved steadily from epoch 1 to epoch 50, with test accuracy increasing from 46.05% to around 80%, showing effective learning and feature extraction.
* Test loss generally decreased from 0.997 at epoch 1 to its lowest stable region around later epochs, indicating improved prediction confidence.
* Test accuracy improved rapidly during early training, crossing 70% by epoch 23 and reaching above 80% after epoch 47.
* Test F1 followed a similar trend, increasing from 0.46 at epoch 1 to above 0.80 in the later training stages.
* Several epochs showed instability where test loss increased while accuracy remained relatively high, indicating prediction confidence fluctuations.
* Epochs 20, 30, 40, 60, 70, and 80 showed performance drops, suggesting temporary generalization instability.
* The model achieved strong generalization in the later training phase, with test accuracy consistently around 78–82% after epoch 90.
* The highest test performance occurred near the final training stages, showing that the model continued improving after the training-validation optimization phase.
* Test precision remained consistently higher than test recall in many epochs, indicating the model was slightly more conservative in its predictions.
* The test metrics closely matched the validation performance, indicating that the model generalized well to unseen data.
* The best test results were achieved at epoch 99, where the model reached the highest test accuracy, recall, and F1 score.
* Epoch 100 showed a slight reduction in all major test metrics compared with epoch 99, indicating that additional training did not improve generalization.

The model demonstrated strong generalization performance on the test set. Starting from 46.05% accuracy at epoch 1, it progressively improved and reached above 80% accuracy in the later training stages. Although some fluctuations occurred due to unstable optimization, the overall trend showed continuous improvement. The model achieved its strongest test performance at epoch 99 with the highest accuracy and F1 score, while epoch 100 showed slight degradation, suggesting that epoch 99 provides the best balance between learning and generalization.

<b>Best Epoch 99</b>

* Loss = 0.6653637610582117
* Accuracy = 0.8223684430122375
* Precision = 0.824786365032196
* Recall = 0.823800265789032
* F1-Score = 0.8223478198051453